
# Generative AI Assessment — Local RAG System (Ollama + Llama 3.2)
---


A full-fledged Retrieval-Augmented Generation (RAG) application that answers questions grounded in a set of PDF documents, running on **Ollama** with the **Llama 3.2** model.

This notebook is self-contained and designed to run on **Google Colab**. It does not generate sample PDFs — upload at least 3 of your own PDF files to `data/pdfs/` (see the upload cell below) before running ingestion.


**Pipeline**: PDFs → chunking → Ollama embeddings → Chroma vector store → retrieval → `llama3.2` generation with cited sources.


## 1. Install Ollama and pull models

Colab runs Linux, so we install Ollama via its official install script, start the server in the background, then pull `llama3.2` (LLM) and `nomic-embed-text` (embeddings).

In [1]:
# 1. Update package list and install zstd
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Run the official Ollama installation script
!curl -fsSL https://ollama.com/install.sh | sh


Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [72.8 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,847 kB]
Hit:8 http://archive.ubuntu.com/ubuntu noble InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [2,990 kB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,242 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:13 https://ppa.launchpadcontent.net/graphi

In [2]:
import subprocess

import time

# Start the Ollama server as a background process

ollama_process = subprocess.Popen(

    ["ollama", "serve"],

    stdout=subprocess.DEVNULL,

    stderr=subprocess.DEVNULL,

)

time.sleep(5)

print("Ollama server started with PID:", ollama_process.pid)

Ollama server started with PID: 1702


In [4]:
!ollama pull llama3.2
!ollama pull nomic-embed-text


## 2. Install Python dependencies

In [5]:
!pip install -q langchain langchain-community langchain-ollama langchain-chroma chromadb pypdf python-dotenv
!pip install -U opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions -q




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 6.1 MB/s eta 0:00:00


In [6]:


import os

from pathlib import Path

BASE_DIR = Path.cwd()

print("Base dir:", BASE_DIR)

Base dir: /content


## 3. Configuration

Central settings for paths, models, chunking, and retrieval

In [7]:


import os

from pathlib import Path


# --- Directories ---

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data" / "pdfs"

VECTOR_STORE_DIR = BASE_DIR / "vector_store"

COLLECTION_NAME = "genai_rag_collection"


# --- Ollama models (must be pulled locally: `ollama pull <model>`) ---

LLM_MODEL = os.getenv("LLM_MODEL", "llama3.2")

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "nomic-embed-text")

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")


# --- Chunking ---

CHUNK_SIZE = 1000

CHUNK_OVERLAP = 150


# --- Retrieval ---

TOP_K = 4


# --- Generation ---

TEMPERATURE = 0.2


DATA_DIR.mkdir(parents=True, exist_ok=True)

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)

print("Vector store dir:", VECTOR_STORE_DIR)

Data dir: /content/data/pdfs
Vector store dir: /content/vector_store


## 4. Upload your PDFs

Upload **at least 3 PDF files**. They will be saved into `data/pdfs/`.

In [8]:
from google.colab import files

uploaded = files.upload()

for filename, content in uploaded.items():

    dest = DATA_DIR / filename

    with open(dest, "wb") as f:

        f.write(content)

    print(f"Saved {dest}")

Saving 52275401_HCL Technologies Ltd._Form16PartA(Traces)_2025_1.pdf to 52275401_HCL Technologies Ltd._Form16PartA(Traces)_2025_1.pdf
Saving 52275401_HCL Technologies Ltd._Form16_PartB_2025_1.pdf to 52275401_HCL Technologies Ltd._Form16_PartB_2025_1.pdf
Saving 52275401_HCL Technologies Ltd._Form16_2024_1.pdf to 52275401_HCL Technologies Ltd._Form16_2024_1.pdf
Saved /content/data/pdfs/52275401_HCL Technologies Ltd._Form16PartA(Traces)_2025_1.pdf
Saved /content/data/pdfs/52275401_HCL Technologies Ltd._Form16_PartB_2025_1.pdf
Saved /content/data/pdfs/52275401_HCL Technologies Ltd._Form16_2024_1.pdf


In [9]:
pdf_paths = sorted(DATA_DIR.glob("*.pdf"))

print(f"Found {len(pdf_paths)} PDF(s):")

for p in pdf_paths:

    print(" -", p.name)

assert len(pdf_paths) >= 3, "Please upload at least 3 PDF files before continuing."

Found 3 PDF(s):
 - 52275401_HCL Technologies Ltd._Form16PartA(Traces)_2025_1.pdf
 - 52275401_HCL Technologies Ltd._Form16_2024_1.pdf
 - 52275401_HCL Technologies Ltd._Form16_PartB_2025_1.pdf


## 5. Ingestion — load, split, embed, and index PDFs

loads each PDF page-by-page, splits into overlapping chunks, embeds with the local `nomic-embed-text` Ollama model, and persists vectors into a Chroma collection.

In [10]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import OllamaEmbeddings

from langchain_chroma import Chroma


def load_pdfs(data_dir: Path):

    pdf_paths = sorted(data_dir.glob("*.pdf"))

    if not pdf_paths:

        raise FileNotFoundError(f"No PDF files found in {data_dir}.")

    if len(pdf_paths) < 3:

        print(f"Warning: only {len(pdf_paths)} PDF(s) found; the assessment expects at least 3.")

    documents = []

    for pdf_path in pdf_paths:

        loader = PyPDFLoader(str(pdf_path))

        docs = loader.load()

        for doc in docs:

            doc.metadata["source"] = pdf_path.name

        documents.extend(docs)

        print(f"Loaded {len(docs)} page(s) from {pdf_path.name}")

    return documents

def split_documents(documents):

    splitter = RecursiveCharacterTextSplitter(

        chunk_size=CHUNK_SIZE,

        chunk_overlap=CHUNK_OVERLAP,

        separators=["\n\n", "\n", ". ", " ", ""],

    )

    chunks = splitter.split_documents(documents)

    print(f"Split into {len(chunks)} chunk(s)")

    return chunks


def build_vector_store(chunks):

    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)

    vector_store = Chroma.from_documents(

        documents=chunks,

        embedding=embeddings,

        collection_name=COLLECTION_NAME,

        persist_directory=str(VECTOR_STORE_DIR),

    )

    print(f"Persisted vector store to {VECTOR_STORE_DIR}")

    return vector_store


documents = load_pdfs(DATA_DIR)

chunks = split_documents(documents)

vector_store = build_vector_store(chunks)

print("Ingestion complete.")

/tmp/ipykernel_435/2533821214.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 2 page(s) from 52275401_HCL Technologies Ltd._Form16PartA(Traces)_2025_1.pdf
Loaded 3 page(s) from 52275401_HCL Technologies Ltd._Form16_2024_1.pdf
Loaded 3 page(s) from 52275401_HCL Technologies Ltd._Form16_PartB_2025_1.pdf
Split into 25 chunk(s)
Persisted vector store to /content/vector_store
Ingestion complete.


## 6. RAG chain — retrieval + prompt + generation

Retrieves relevant chunks from Chroma and generates an answer using `llama3.2` via `ChatOllama`, with source citations.

In [11]:
from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser

from langchain_core.runnables import RunnablePassthrough


SYSTEM_PROMPT = (

    "You are a helpful assistant answering questions using ONLY the provided context "

    "extracted from a set of PDF documents. "

    "If the answer cannot be found in the context, say you don't know instead of "

    "making up an answer. Always be concise and cite the source file name(s) you used."

)


PROMPT_TEMPLATE = ChatPromptTemplate.from_messages([

    ("system", SYSTEM_PROMPT),

    ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"),

])


def format_docs(docs):

    return "\n\n".join(

        f"[Source: {d.metadata.get('source', 'unknown')} | page {d.metadata.get('page', '?')}]\n{d.page_content}"

        for d in docs

    )


class RAGPipeline:

    def __init__(self):

        self.embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)

        self.vector_store = Chroma(

            collection_name=COLLECTION_NAME,

            embedding_function=self.embeddings,

            persist_directory=str(VECTOR_STORE_DIR),

        )

        self.retriever = self.vector_store.as_retriever(search_kwargs={"k": TOP_K})

        self.llm = ChatOllama(model=LLM_MODEL, base_url=OLLAMA_BASE_URL, temperature=TEMPERATURE)


        self.chain = (

            {"context": self.retriever | format_docs, "question": RunnablePassthrough()}

            | PROMPT_TEMPLATE

            | self.llm

            | StrOutputParser()

        )


    def retrieve(self, question: str):

        return self.retriever.invoke(question)


    def answer(self, question: str) -> str:

        return self.chain.invoke(question)


    def answer_with_sources(self, question: str):

        docs = self.retrieve(question)

        answer = self.chain.invoke(question)

        sources = sorted({d.metadata.get("source", "unknown") for d in docs})

        return answer, sources



pipeline = RAGPipeline()

print("RAG pipeline ready.")

RAG pipeline ready.


## 7. Ask questions

Run this cell repeatedly with different questions grounded in your uploaded PDFs.

In [12]:

question = "What is this document collection about?"  # <-- edit your question here


answer, sources = pipeline.answer_with_sources(question)

print("Question:", question)

print("\nAnswer:", answer)

print("\nSources:", ", ".join(sources) if sources else "none")

Question: What is this document collection about?

Answer: This document collection appears to be about tax certificates, specifically Form 16, which is used to certify the tax deductions and deposits made by an employee to the Income Tax Department. The documents provide details about the tax deductions, deposits, and other income reported by the employee, as well as the tax payable and other relevant information.

Sources: 52275401_HCL Technologies Ltd._Form16PartA(Traces)_2025_1.pdf, 52275401_HCL Technologies Ltd._Form16_2024_1.pdf, 52275401_HCL Technologies Ltd._Form16_PartB_2025_1.pdf
